# preEHA — Preparar datos para Event History Analysis

Este notebook hace únicamente **preparación de datos**.

Su producto final es una base en la que cada gabinete tiene:

- una duración observada;
- un indicador de si la caída fue observada o si el caso quedó censurado;
- las variables políticas que utilizaremos después.

El análisis de supervivencia se realiza en el notebook **EHA**.

## preEHA-01 — Leer los datos sin procesar

In [ ]:
import pandas as pd

# Datos originales del curso, publicados en el repositorio de GitHub
url_raw = "https://github.com/doctorado-cuanticp/eha/raw/main/gabinetes_sin_procesar.csv"

gabinetes = pd.read_csv(url_raw)
gabinetes.head()
# Debe mostrar 8 columnas, incluyendo fecha_inicio, fecha_fin_observacion,
# situacion_final y tipo_gobierno (las que usaremos para construir las
# variables del análisis de supervivencia).

## preEHA-02 — Revisar fechas y situación final

In [ ]:
gabinetes[
    ["fecha_inicio", "fecha_fin_observacion", "situacion_final"]
].head()
# Revisamos estas tres columnas porque de ellas depende todo el diseño:
# las dos fechas dan la duración, y situacion_final distingue caída de censura.

La fecha final no significa siempre que el gabinete haya caído.

Si `situacion_final` es `vigente_al_cierre`, solo sabemos que el gabinete sobrevivió hasta esa fecha.
Ese caso será **censurado a la derecha**.

## preEHA-03 — Construir duración y evento

In [ ]:
gabinetes["fecha_inicio"] = pd.to_datetime(gabinetes["fecha_inicio"])
gabinetes["fecha_fin_observacion"] = pd.to_datetime(gabinetes["fecha_fin_observacion"])

gabinetes["duracion_meses"] = (
    (gabinetes["fecha_fin_observacion"] - gabinetes["fecha_inicio"]).dt.days
    / 30.4375
).round(1)

gabinetes["caida_evento"] = (
    gabinetes["situacion_final"] == "caida"
).astype(int)

gabinetes[
    ["duracion_meses", "caida_evento"]
].head()
# Dividimos entre 30.4375 (promedio de días por mes) para expresar la duración
# en meses, no en días.
# caida_evento = 1 si el gabinete cayó; 0 si el caso terminó censurado
# ("vigente_al_cierre"), sin que eso signifique que no haya información útil.

## preEHA-04 — Codificar gobierno mayoritario

In [ ]:
gabinetes["gobierno_mayoritario"] = (
    gabinetes["tipo_gobierno"] == "mayoritario"
).astype(int)

gabinetes[
    ["tipo_gobierno", "gobierno_mayoritario"]
].head()
# gobierno_mayoritario = 1 si tipo_gobierno es "mayoritario", 0 si es "minoritario".
# Es la variable que EHA-04 usará para comparar las dos curvas de supervivencia.

## preEHA-05 — Guardar la base lista para EHA

In [ ]:
# fecha_inicio y fecha_fin_observacion ya cumplieron su función: sirvieron
# para calcular duracion_meses. El notebook de análisis (EHA) no vuelve a
# usarlas, así que no las incluimos en la base final ni necesitamos pickle
# para preservar su tipo datetime: un CSV es suficiente y más simple.
eha = gabinetes[
    [
        "id_gabinete",
        "duracion_meses",
        "caida_evento",
        "gobierno_mayoritario",
        "fragmentacion",
        "crecimiento_pbi"
    ]
].copy()

eha.to_csv("eha_procesada.csv", index=False)

eha.head()

## Producto de preEHA

La base `eha_procesada.csv` está lista para análisis de supervivencia.

El notebook **EHA** no volverá a calcular duración, censura ni codificaciones.